# 09 — Robust edge validation

This notebook checks whether the strategy found in notebook 08 is robust or likely to be noise.

Checks included:
1. Chronological train / validation / test split.
2. Breakdown by outcome: away / draw / home.
3. Breakdown by odds buckets.
4. Breakdown by tournament label, where available.
5. Bootstrap confidence interval for return.
6. Permutation test.
7. Conservative strategy shortlist.
8. Paper-only decision report.

API calls: 0.
Real-money stake: 0.


In [ ]:
# Cell 1. Imports and helpers.

from pathlib import Path
import json
import zipfile
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

DATA_DIR = Path("data")
PROCESSED_DIR = DATA_DIR / "processed"
IN_DIR = PROCESSED_DIR / "08_edge_selector_grid"
OUT_DIR = PROCESSED_DIR / "09_robust_edge_validation"

OUT_DIR.mkdir(parents=True, exist_ok=True)

RUN_TIMESTAMP_UTC = pd.Timestamp.utcnow().isoformat()
SEED = 42
rng = np.random.default_rng(SEED)

def save_json(path, payload):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(payload, f, indent=2, ensure_ascii=False, default=str)
    return path

def bootstrap_mean_ci(values, n_boot=5000, seed=SEED):
    values = np.asarray(values, dtype=float)
    values = values[~np.isnan(values)]

    if len(values) == 0:
        return {
            "mean": np.nan,
            "ci_low": np.nan,
            "ci_high": np.nan,
        }

    local_rng = np.random.default_rng(seed)
    means = []

    for _ in range(n_boot):
        sample = local_rng.choice(values, size=len(values), replace=True)
        means.append(np.mean(sample))

    means = np.asarray(means)

    return {
        "mean": float(np.mean(values)),
        "ci_low": float(np.quantile(means, 0.025)),
        "ci_high": float(np.quantile(means, 0.975)),
    }

def summarize_picks(df, name):
    if len(df) == 0:
        return {
            "group": name,
            "n_picks": 0,
            "hit_rate": np.nan,
            "roi_flat": np.nan,
            "profit_flat": 0.0,
            "avg_odds": np.nan,
            "avg_clv_proxy_odds": np.nan,
            "median_clv_proxy_odds": np.nan,
            "avg_edge": np.nan,
            "avg_ev": np.nan,
        }

    return {
        "group": name,
        "n_picks": int(len(df)),
        "hit_rate": float(df["selection_won"].mean()),
        "roi_flat": float(df["profit_1usd"].mean()),
        "profit_flat": float(df["profit_1usd"].sum()),
        "avg_odds": float(df["best_odds_24h"].mean()),
        "avg_clv_proxy_odds": float(df["clv_proxy_odds"].mean()),
        "median_clv_proxy_odds": float(df["clv_proxy_odds"].median()),
        "avg_edge": float(df["blend_edge"].mean()),
        "avg_ev": float(df["blend_ev"].mean()),
    }

print("Setup OK.")


In [ ]:
# Cell 2. Load 08 outputs.

paths = {
    "best_picks": IN_DIR / "best_candidate_picks_all_splits.csv",
    "grid": IN_DIR / "strategy_grid_validation_test.csv",
    "candidates": IN_DIR / "candidate_strategies.csv",
    "all_selections": IN_DIR / "all_selection_rows.csv",
    "best_params": IN_DIR / "best_candidate_params.json",
}

missing = [name for name, path in paths.items() if not path.exists()]

if missing:
    raise FileNotFoundError(
        f"Missing 08 outputs: {missing}. Run 08 first."
    )

best_picks = pd.read_csv(paths["best_picks"])
grid = pd.read_csv(paths["grid"])
candidates = pd.read_csv(paths["candidates"])
all_selections = pd.read_csv(paths["all_selections"])

with open(paths["best_params"], "r", encoding="utf-8") as f:
    best_params = json.load(f)

for df in [best_picks, all_selections]:
    if "date" in df.columns:
        df["date"] = pd.to_datetime(df["date"], errors="coerce")

print("best params:", best_params)
print("best picks:", best_picks.shape)
print("grid:", grid.shape)
print("candidates:", candidates.shape)

display(best_picks.head())


In [ ]:
# Cell 3. Overall and split summary.

summary_rows = []

summary_rows.append(summarize_picks(best_picks, "all"))

for split, part in best_picks.groupby("split"):
    summary_rows.append(summarize_picks(part, f"split_{split}"))

overall_summary = pd.DataFrame(summary_rows)

overall_summary.to_csv(
    OUT_DIR / "overall_and_split_summary.csv",
    index=False,
)

display(overall_summary)


In [ ]:
# Cell 4. Selection and odds bucket diagnostics.

diag_rows = []

for selection, part in best_picks.groupby("selection"):
    diag_rows.append(summarize_picks(part, f"selection_{selection}"))

bins = [1.0, 1.8, 2.5, 4.0, 6.0, 10.0, 100.0]
labels = ["1.0-1.8", "1.8-2.5", "2.5-4.0", "4.0-6.0", "6.0-10.0", "10+"]

best_picks["odds_bucket"] = pd.cut(
    best_picks["best_odds_24h"],
    bins=bins,
    labels=labels,
    include_lowest=True,
)

for bucket, part in best_picks.groupby("odds_bucket"):
    diag_rows.append(summarize_picks(part, f"odds_{bucket}"))

for split, split_part in best_picks.groupby("split"):
    for selection, part in split_part.groupby("selection"):
        diag_rows.append(
            summarize_picks(part, f"split_{split}_selection_{selection}")
        )

diagnostics = pd.DataFrame(diag_rows)

diagnostics.to_csv(
    OUT_DIR / "selection_odds_bucket_diagnostics.csv",
    index=False,
)

display(diagnostics)


In [ ]:
# Cell 5. Bootstrap ROI confidence intervals.

boot_rows = []

groups = {
    "all": best_picks,
}

for split, part in best_picks.groupby("split"):
    groups[f"split_{split}"] = part

for selection, part in best_picks.groupby("selection"):
    groups[f"selection_{selection}"] = part

for name, part in groups.items():
    if len(part) == 0:
        continue

    roi_ci = bootstrap_mean_ci(part["profit_1usd"].to_numpy())
    clv_ci = bootstrap_mean_ci(part["clv_proxy_odds"].to_numpy())

    boot_rows.append({
        "group": name,
        "n_picks": len(part),
        "roi_mean": roi_ci["mean"],
        "roi_ci_low": roi_ci["ci_low"],
        "roi_ci_high": roi_ci["ci_high"],
        "clv_mean": clv_ci["mean"],
        "clv_ci_low": clv_ci["ci_low"],
        "clv_ci_high": clv_ci["ci_high"],
    })

bootstrap_report = pd.DataFrame(boot_rows)

bootstrap_report.to_csv(
    OUT_DIR / "bootstrap_roi_clv_report.csv",
    index=False,
)

display(bootstrap_report)


In [ ]:
# Cell 6. Permutation test for ROI.

# Null: selected rows have no special edge; outcomes are randomly shuffled
# across selected picks while odds stay fixed.

perm_rows = []

for group_name, part in groups.items():
    if len(part) < 5:
        continue

    observed_profit = float(part["profit_1usd"].mean())
    profits = []

    odds = part["best_odds_24h"].to_numpy()
    wins = part["selection_won"].to_numpy()

    for _ in range(5000):
        shuffled = rng.permutation(wins)
        profit = np.where(shuffled == 1, odds - 1.0, -1.0)
        profits.append(np.mean(profit))

    profits = np.asarray(profits)

    p_value = float((profits >= observed_profit).mean())

    perm_rows.append({
        "group": group_name,
        "n_picks": len(part),
        "observed_roi": observed_profit,
        "null_mean_roi": float(profits.mean()),
        "null_95_low": float(np.quantile(profits, 0.025)),
        "null_95_high": float(np.quantile(profits, 0.975)),
        "p_value_profit_ge_observed": p_value,
    })

permutation_report = pd.DataFrame(perm_rows)

permutation_report.to_csv(
    OUT_DIR / "permutation_roi_report.csv",
    index=False,
)

display(permutation_report)


In [ ]:
# Cell 7. Conservative shortlist from all candidate strategies.

# We only accept strategies that:
# - have at least 15 validation picks;
# - have at least 15 test picks;
# - validation ROI > 0;
# - test ROI > 0;
# - test average CLV proxy is not too negative;
# - odds are capped below or equal 6.

if len(candidates) == 0:
    robust_shortlist = pd.DataFrame()
else:
    robust_shortlist = candidates[
        (candidates["val_n_picks"] >= 15)
        & (candidates["test_n_picks"] >= 15)
        & (candidates["val_roi_flat"] > 0)
        & (candidates["test_roi_flat"] > 0)
        & (candidates["test_avg_clv_proxy_odds"] > -0.02)
        & (candidates["max_odds"] <= 6.0)
    ].copy()

    if len(robust_shortlist) > 0:
        robust_shortlist = robust_shortlist.sort_values(
            [
                "test_avg_clv_proxy_odds",
                "test_roi_flat",
                "val_roi_flat",
            ],
            ascending=[False, False, False],
        )

robust_shortlist.to_csv(
    OUT_DIR / "robust_strategy_shortlist.csv",
    index=False,
)

display(robust_shortlist.head(30))
print("robust shortlist rows:", len(robust_shortlist))


In [ ]:
# Cell 8. Decision report.

red_flags = []

test_summary = overall_summary[
    overall_summary["group"] == "split_test"
]

if len(test_summary) > 0:
    test_row = test_summary.iloc[0]

    if test_row["n_picks"] < 30:
        red_flags.append("test sample below 30 picks")

    if test_row["avg_clv_proxy_odds"] < 0:
        red_flags.append("test average CLV proxy is negative")

    if test_row["roi_flat"] <= 0:
        red_flags.append("test ROI is not positive")
else:
    red_flags.append("no test split summary")

train_summary = overall_summary[
    overall_summary["group"] == "split_train"
]

if len(train_summary) > 0:
    train_row = train_summary.iloc[0]
    if train_row["roi_flat"] < 0:
        red_flags.append("train ROI is negative")

if len(robust_shortlist) == 0:
    red_flags.append("no robust strategy passed conservative filters")

decision = {
    "real_money_allowed": False,
    "paper_tracking_allowed": True,
    "red_flags": red_flags,
    "recommended_action": (
        "Keep as paper strategy only. "
        "Collect more historical odds and current snapshots. "
        "Require non-negative CLV before any real-money discussion."
    ),
}

save_json(
    OUT_DIR / "decision_report.json",
    decision,
)

display(pd.DataFrame([decision]))


In [ ]:
# Cell 9. Save zip.

summary = {
    "run_timestamp_utc": RUN_TIMESTAMP_UTC,
    "best_params_from_08": best_params,
    "best_picks": int(len(best_picks)),
    "grid_rows": int(len(grid)),
    "candidate_rows": int(len(candidates)),
    "robust_shortlist_rows": int(len(robust_shortlist)),
    "real_money_allowed": False,
    "note": "Robustness validation only. No forecasting advice.",
}

save_json(
    OUT_DIR / "summary.json",
    summary,
)

zip_path = PROCESSED_DIR / "09_robust_edge_validation_report_bundle.zip"

with zipfile.ZipFile(
    zip_path,
    "w",
    compression=zipfile.ZIP_DEFLATED,
) as zf:
    for file in OUT_DIR.rglob("*"):
        if file.is_file():
            zf.write(file, arcname=file.relative_to(OUT_DIR))

display(pd.DataFrame([summary]))

print("Created:", zip_path.resolve())
print("Report bundle created.")
